# Filtrando e agrupando dados com pandas: emissões de gases de efeito estufa (SEEG)
Seleção com `loc` e `isin`, reestruturação com `melt` e `pivot_table`, agrupamentos com `groupby` (inclusive multi-índice com `xs`, `idxmax` e `swaplevel`) e junção com a população dos estados para calcular a emissão per capita, com gráficos em pandas e Plotly.

**Origem:** Alura — curso de pandas sobre seleção e agrupamento de dados (formação Python para Data Science).

**Dados:** envie para `/content/` no Colab antes de executar:
- `1-SEEG10_GERAL-BR_UF_2022.10.27-FINAL-SITE.xlsx` — planilha do SEEG (Sistema de Estimativas de Emissões e Remoções de Gases de Efeito Estufa, https://seeg.eco.br), aba `GEE Estados`.
- `POP2022_Municipios.xls` — estimativa da população dos municípios em 2022 (IBGE).

**Índice**
1. Carregando os dados
2. Filtrando emissões e remoções
3. Seleções com `loc` e `isin`
4. Reestruturando a tabela com `melt`
5. Agrupando por gás
6. Agrupando por setor
7. Multi-índice: gás e setor
8. Evolução temporal das emissões
9. Tarefas: `pivot_table` e emissões por estado
10. Emissão per capita por estado

In [ ]:
# Setup (openpyxl e xlrd, usados por read_excel, e plotly já vêm no Colab)
import pandas as pd
import plotly.express as px

## 1. Carregando os dados
A aba `GEE Estados` tem uma linha por combinação de setor, categoria, gás e estado, e uma coluna por ano (1970 a 2021).

In [ ]:
df = pd.read_excel('/content/1-SEEG10_GERAL-BR_UF_2022.10.27-FINAL-SITE.xlsx', sheet_name='GEE Estados')
df.head()

## 2. Filtrando emissões e remoções
A coluna `Emissão / Remoção / Bunker` indica se a linha é uma emissão, uma remoção (valores negativos) ou bunker. Conferimos as remoções e depois mantemos apenas as emissões.

In [ ]:
df['Emissão / Remoção / Bunker'].unique()   # valores únicos da coluna

In [ ]:
# Maior valor de cada ano entre as linhas de remoção (devem ser <= 0)
df.loc[df['Emissão / Remoção / Bunker'].isin(['Remoção NCI', 'Remoção']), 1970:2021].max()

In [ ]:
# Estados com remoção NCI
df.loc[df['Emissão / Remoção / Bunker'] == 'Remoção NCI', 'Estado'].unique()

In [ ]:
df = df[df['Emissão / Remoção / Bunker'] == 'Emissão']
df = df.drop(columns=['Emissão / Remoção / Bunker'])

print(df['Nível 1 - Setor'].unique())
df['Estado'].unique()

## 3. Seleções com `loc` e `isin`
`isin` testa se o valor está numa lista; `loc[linhas, colunas]` combina condições sobre as linhas com a escolha das colunas.

In [ ]:
# Linhas dos estados do Sul
df[df['Estado'].isin(['PR', 'SC', 'RS'])]

In [ ]:
# Maior emissão por mudança de uso da terra no Amazonas em 2021
df.loc[(df['Nível 1 - Setor'] == 'Mudança de Uso da Terra e Floresta') &
       (df['Estado'] == 'AM'), 2021].max()

In [ ]:
# Quantidade de registros de NOx da indústria química em 2017
df.loc[(df['Nível 2'] == 'Indústria Química') &
       (df['Gás'] == 'NOx (t)'), 2017].count()

## 4. Reestruturando a tabela com `melt`
`melt` transforma as colunas de anos em linhas: cada linha passa a ter um `Ano` e o valor de `Emissão`, o formato adequado para agrupar por ano.

In [ ]:
colunas_escolhidas = list(df.loc[:, 'Nível 1 - Setor':'Produto'].columns)   # colunas descritivas
colunas_pivot = list(df.loc[:, 1970:2021].columns)                           # colunas de anos

df = df.melt(id_vars=colunas_escolhidas, value_vars=colunas_pivot,
             var_name='Ano', value_name='Emissão')

df.info()

## 5. Agrupando por gás
`groupby` separa os dados em grupos; `get_group` mostra um grupo específico. Principais agregações:

| Método | O que calcula |
|---|---|
| `mean()` | média dos grupos |
| `sum()` | soma dos valores do grupo |
| `size()` | tamanho dos grupos |
| `count()` | contagem (valores não nulos) do grupo |
| `std()` | desvio padrão dos grupos |
| `var()` | variância dos grupos |
| `sem()` | erro padrão da média dos grupos |
| `describe()` | estatísticas descritivas |
| `first()` / `last()` | primeiro / último valor do grupo |
| `nth()` | n-ésimo valor do grupo |
| `min()` / `max()` | mínimo / máximo do grupo |

In [ ]:
df.groupby('Gás').get_group('CO2 (t)').head()

In [ ]:
df.groupby('Gás').get_group('CO2 (t)').count()

In [ ]:
emissao_por_gas = df.groupby('Gás')[['Emissão']].sum().sort_values('Emissão', ascending=True)
emissao_por_gas

In [ ]:
emissao_por_gas.plot(kind='barh',
                     figsize=(10, 6),
                     xlabel='Emissão (t)',
                     ylabel='Gás')

In [ ]:
# Participação do CO2 no total emitido em toneladas, de 1970 a 2021.
# As linhas "CO2e" são o total convertido em CO2 equivalente (somam todos os gases),
# por isso ficam fora do denominador.
emissao_gases = emissao_por_gas[~emissao_por_gas.index.str.contains('CO2e')]
participacao_co2 = emissao_gases.loc['CO2 (t)', 'Emissão'] / emissao_gases['Emissão'].sum() * 100
print(f'A emissão de CO2 corresponde a {participacao_co2:.2f} % da emissão total de gases de efeito estufa no Brasil de 1970 a 2021.')

## 6. Agrupando por setor

In [ ]:
df.groupby('Nível 1 - Setor').get_group('Agropecuária').head()

In [ ]:
# Emissão média por setor em 2021
df[df['Ano'] == 2021].groupby('Nível 1 - Setor')[['Emissão']].mean()

In [ ]:
# Emissão total por setor (1970-2021)
df.groupby('Nível 1 - Setor')[['Emissão']].sum().sort_values('Emissão', ascending=False)

## 7. Multi-índice: gás e setor
Agrupar por duas colunas cria um índice com dois níveis. `xs` seleciona um valor de um nível, `idxmax` retorna o índice do maior valor e `swaplevel` troca a ordem dos níveis.

In [ ]:
gas_por_setor = df.groupby(['Gás', 'Nível 1 - Setor'])[['Emissão']].sum()
gas_por_setor

In [ ]:
gas_por_setor.xs('CH4 (t)', level=0)

In [ ]:
gas_por_setor.xs(('CH4 (t)', 'Mudança de Uso da Terra e Floresta'), level=[0, 1])

In [ ]:
# Setor que mais emite CO2
gas_por_setor.xs('CO2 (t)', level=0).idxmax()

In [ ]:
# Para cada gás, o setor que mais emite e a quantidade emitida
valores_max = gas_por_setor.groupby(level=0)['Emissão'].max().values
tabela_sumarizada = gas_por_setor.groupby(level=0).idxmax()
tabela_sumarizada.insert(1, 'Quantidade de emissão', valores_max)
tabela_sumarizada

In [ ]:
# Invertendo os níveis: para cada setor, o gás mais emitido
gas_por_setor.swaplevel(0, 1).groupby(level=0).idxmax()

## 8. Evolução temporal das emissões

In [ ]:
df.groupby('Ano')[['Emissão']].mean().plot(figsize=(10, 6))

In [ ]:
# Ano com maior emissão média
df.groupby('Ano')[['Emissão']].mean().idxmax()

In [ ]:
# Média anual por gás, em formato longo e depois em tabela (um gás por coluna)
media_emissao_anual = df.groupby(['Ano', 'Gás'])[['Emissão']].mean().reset_index()
media_emissao_anual

In [ ]:
media_emissao_anual = media_emissao_anual.pivot_table(index='Ano', columns='Gás', values='Emissão')
media_emissao_anual.plot(subplots=True, figsize=(10, 40))

## 9. Tarefas: `pivot_table` e emissões por estado
`pivot_table` faz o caminho inverso do `melt`: transforma os valores de uma coluna (aqui, `Ano`) em colunas.

In [ ]:
df10 = pd.DataFrame({
    'Ano': [1970, 1970, 1970, 1971, 1971, 1971, 1972, 1972, 1972],
    'Gás': ['C2F6 (t)', 'CF4 (t)', 'CH4 (t)', 'C2F6 (t)', 'CF4 (t)', 'CH4 (t)', 'C2F6 (t)', 'CF4 (t)', 'CH4 (t)'],
    'Emissão': [0.232610, 3.557421, 1471.024024, 0.33461, 5.1173, 1525.5, 0.40468, 6.18902, 1584.7195]
})

df10.pivot_table(index='Gás', columns='Ano', values='Emissão')

In [ ]:
emissoes_estados_setor = df.groupby(['Estado', 'Nível 1 - Setor'])[['Emissão']].sum()
emissoes_estados_setor

In [ ]:
# Emissões do setor de Energia por estado
emissoes_estados_setor.xs('Energia', level=1)

In [ ]:
# Setor que mais emite em Minas Gerais
emissoes_estados_setor.xs('MG', level=0).idxmax()

In [ ]:
# Setor que mais emite em cada estado
emissoes_estados_setor.groupby(level=0).idxmax()

In [ ]:
# Maior emissão individual por setor e estado; para cada estado, o setor com a maior delas
emissao_ativ_econ = df.groupby(['Nível 1 - Setor', 'Estado'])[['Emissão']].max().sort_values('Emissão', ascending=False)
emissao_ativ_econ.groupby(level=1).idxmax()

## 10. Emissão per capita por estado
A planilha do IBGE tem a população por município. Alguns valores vêm como texto, com separador de milhar e uma nota entre parênteses (ex.: `"12.345(1)"`); limpamos esses valores, somamos por UF e juntamos com a emissão de 2021 de cada estado.

In [ ]:
df_pop = pd.read_excel('/content/POP2022_Municipios.xls', header=1, skipfooter=34)
df_pop.head()

In [ ]:
# Linhas com nota entre parênteses na população
df_pop[df_pop['POPULAÇÃO'].astype(str).str.contains(r'\(', na=False)]

In [ ]:
# Remove a nota "(n)" e o separador de milhar; os valores que já eram numéricos não são alterados
populacao_estados = df_pop.assign(
    populacao_sem_parenteses=df_pop['POPULAÇÃO'].replace(r'\(\d{1,2}\)', '', regex=True),
    populacao=lambda x: x['populacao_sem_parenteses'].replace(r'\.', '', regex=True)
)
populacao_estados['populacao'] = populacao_estados['populacao'].astype(int)
populacao_estados = populacao_estados.groupby('UF')[['populacao']].sum().reset_index()
populacao_estados

In [ ]:
emissao_estado = df[df['Ano'] == 2021].groupby('Estado')[['Emissão']].sum().reset_index()
emissao_estado

In [ ]:
emissao_estado_populacao = pd.merge(emissao_estado, populacao_estados, left_on='Estado', right_on='UF')
emissao_estado_populacao = emissao_estado_populacao.drop(columns='UF')
emissao_estado_populacao = emissao_estado_populacao.assign(
    emissao_per_capita=emissao_estado_populacao['Emissão'] / emissao_estado_populacao['populacao']
).sort_values('emissao_per_capita', ascending=False)
emissao_estado_populacao

In [ ]:
px.scatter(data_frame=emissao_estado_populacao, x='populacao', y='Emissão',
           text='Estado', size='emissao_per_capita')

In [ ]:
px.bar(data_frame=emissao_estado_populacao, x='Estado', y='emissao_per_capita')